In [1]:
 import sys
sys.path.append("..")  # so Python can find the 'app' package one level up

from app.ingestion.parsers import parse_pdf
from app.chunking.strategies import chunk_fixed_size, chunk_recursive

In [2]:
text = parse_pdf("../../data/sample_docs/test.pdf")
print("Total characters:", len(text))

fixed_chunks = chunk_fixed_size(text, chunk_size=500, overlap=50)
recursive_chunks = chunk_recursive(text, chunk_size=500, overlap=50)

print("\nFixed-size chunk count:", len(fixed_chunks))
print("Recursive chunk count:", len(recursive_chunks))

print("\n--- RECURSIVE CHUNK 1 ---")
print(recursive_chunks[0])

if len(recursive_chunks) > 1:
    print("\n--- RECURSIVE CHUNK 2 ---")
    print(recursive_chunks[1])

Total characters: 26764

Fixed-size chunk count: 15
Recursive chunk count: 16

--- RECURSIVE CHUNK 1 ---
[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring Legal Fine Print
Each entry in this book contains material from Wikipedia, although the text we use 
may not represent the current version of any article. The URL at the bottom of each 
page will direct you to the source Wikipedia article; use the article’s History tab to 
find a list of contributors.
All material in this book that is taken from Wikipedia is licensed under the Creative 
Commons-Attribution Share Alike 3.0 license.  Here’s a quick human-readable 
summary of your rights to use this content:
You are free:
 to Share—to copy, distribute and transmit the work, and
 to Remix—to adapt the work
Under the following conditions:
 Attribution—You must attribute the work in the manner specified by 
the author or licensor (but not in any way that suggests that they endorse you or 
you

In [3]:
from google import genai as genai_client

client = genai_client.Client(api_key=api_key)

for model in client.models.list():
    if "embedContent" in model.supported_actions:
        print(model.name)

NameError: name 'api_key' is not defined

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
print("Key loaded:", api_key[:8] + "..." if api_key else "NOT FOUND")

In [ ]:
from google import genai as genai_client

client = genai_client.Client(api_key=api_key)

for model in client.models.list():
    if "embedContent" in model.supported_actions:
        print(model.name)

In [4]:
result = client.models.embed_content(
    model="models/gemini-embedding-001",
    contents="This is a test sentence."
)

print("Embedding length:", len(result.embeddings[0].values))
print("First 5 values:", result.embeddings[0].values[:5])

NameError: name 'client' is not defined

In [5]:
from app.embeddings.embedder import embed_text

vec = embed_text("Hello world")
print(len(vec))

3072


In [6]:
from app.chunking.strategies import chunk_semantic

# Take just a small excerpt first - don't run this on the full 26,000-character
# document yet, since semantic chunking is slow (one API call per sentence)
excerpt = text[:2000]  # first 2000 characters only
print("Excerpt being tested:\n", excerpt)
print("\n" + "="*50 + "\n")

semantic_chunks = chunk_semantic(excerpt, similarity_threshold=0.75)

print("Number of semantic chunks:", len(semantic_chunks))
for i, chunk in enumerate(semantic_chunks):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk)

Excerpt being tested:
 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring Legal Fine Print
Each entry in this book contains material from Wikipedia, although the text we use 
may not represent the current version of any article. The URL at the bottom of each 
page will direct you to the source Wikipedia article; use the article’s History tab to 
find a list of contributors.
All material in this book that is taken from Wikipedia is licensed under the Creative 
Commons-Attribution Share Alike 3.0 license.  Here’s a quick human-readable 
summary of your rights to use this content:
You are free:
 to Share—to copy, distribute and transmit the work, and
 to Remix—to adapt the work
Under the following conditions:
 Attribution—You must attribute the work in the manner specified by 
the author or licensor (but not in any way that suggests that they endorse you or 
your use of the work.)
 Share Alike—If you alter, transform, or build upon this wor

In [7]:
semantic_chunks_v2 = chunk_semantic(excerpt, similarity_threshold=0.5)
print("Number of semantic chunks (threshold=0.5):", len(semantic_chunks_v2))
for i, chunk in enumerate(semantic_chunks_v2):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk)

Number of semantic chunks (threshold=0.5): 1

--- CHUNK 1 ---
[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring Legal Fine Print
Each entry in this book contains material from Wikipedia, although the text we use 
may not represent the current version of any article. The URL at the bottom of each 
page will direct you to the source Wikipedia article; use the article’s History tab to 
find a list of contributors. All material in this book that is taken from Wikipedia is licensed under the Creative 
Commons-Attribution Share Alike 3.0 license. Here’s a quick human-readable 
summary of your rights to use this content:
You are free:
 to Share—to copy, distribute and transmit the work, and
 to Remix—to adapt the work
Under the following conditions:
 Attribution—You must attribute the work in the manner specified by 
the author or licensor (but not in any way that suggests that they endorse you or 
your use of the work.)
 Share Alike—If you alte

In [8]:
from app.embeddings.embedder import embed_text
from app.chunking.strategies import _split_into_sentences, _cosine_similarity

sentences = _split_into_sentences(excerpt)
embeddings = [embed_text(s) for s in sentences]

for i in range(1, len(sentences)):
    sim = _cosine_similarity(embeddings[i-1], embeddings[i])
    print(f"{sim:.3f}  |  ...{sentences[i-1][-40:]}  -->  {sentences[i][:40]}...")

0.673  |  ...sent the current version of any article.  -->  The URL at the bottom of each 
page will...
0.717  |  ...ory tab to 
find a list of contributors.  -->  All material in this book that is taken ...
0.735  |  ...ons-Attribution Share Alike 3.0 license.  -->  Here’s a quick human-readable 
summary o...
0.732  |  ...e same, similar or a compatible license.  -->  With the understanding that:
 Waiver—Any...
0.741  |  ...t 
permission from the copyright holder.  -->  Other Rights—In no way are any of the fo...
0.704  |  ...ed, such as publicity or privacy rights.  -->  Notice—For any reuse or distribution, yo...
0.669  |  ... others 
the license terms of this work.  -->  The best way to do this is with a link t...
0.661  |  ...011 Conor Lastowka and 
Josh Fruhlinger.  -->  Copy-edited by Lauren Lastowka
Cover des...
0.610  |  ... who wrote an 
entry that appears in it.  -->  May your citations always be needed....
0.573  |  ...May your citations always be needed.  -->  6
Introducti

In [9]:
semantic_chunks_v3 = chunk_semantic(excerpt, similarity_threshold=0.6)
print("Number of chunks (threshold=0.6):", len(semantic_chunks_v3))
for i, chunk in enumerate(semantic_chunks_v3):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk[:150], "...")

Number of chunks (threshold=0.6): 2

--- CHUNK 1 ---
[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring Legal Fine Print
Each entry in this book contains ma ...

--- CHUNK 2 ---
6
Introduction
Wikip ...


In [1]:
import sys
sys.path.append("..")

from app.embeddings.embedder import embed_text

vec = embed_text("Hello world")
print("Dimensions:", len(vec))

Dimensions: 768


In [2]:
import sys
sys.path.append("..")

from app.ingestion.parsers import parse_pdf
from app.chunking.strategies import chunk_fixed_size
from app.db.vector_store import add_document, store_chunks, search_chunks

# Step 1: Parse (Day 1)
text = parse_pdf("../../data/sample_docs/test.pdf")
print(f"Parsed: {len(text)} characters")

# Step 2: Chunk (Day 2)
chunks = chunk_fixed_size(text, chunk_size=500, overlap=50)
print(f"Chunked: {len(chunks)} chunks")

# Step 3: Store in DB (Day 3)
doc_id = add_document("test.pdf", "pdf")
print(f"Document ID: {doc_id}")

store_chunks(doc_id, chunks, strategy="fixed")

# Step 4: Search
results = search_chunks("what is the formula for variance?", top_k=3)
print(f"\nTop {len(results)} results:")
for r in results:
    print(f"\nSimilarity: {r['similarity']:.3f}")
    print(f"Content: {r['content'][:150]}...")

Parsed: 26764 characters
Chunked: 15 chunks
Document ID: 1
Embedding 15 chunks using strategy 'fixed'...
Stored 15 chunks successfully.

Top 3 results:

Similarity: 0.483
Content:  else is a faithful reproduction of 
the way the entry stood at the moment we or our informants 
encountered it. We hope you will laugh, cry, maybe ev...

Similarity: 0.474
Content:  so-called “Doctor” Dre never 
actually received a PhD.
D.A.R.E. America has its headquarters in 
Inglewood. Despite this, in the 1996 rap 
hit “Calif...

Similarity: 0.473
Content: 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiarized a history report from it, or simply replaced the 
entire text of t...


In [3]:
import sys
sys.path.append("..")

from app.ingestion.parsers import parse_pdf
from app.chunking.strategies import chunk_fixed_size
from app.db.vector_store import add_document, store_chunks

text = parse_pdf("../../data/sample_docs/test.pdf")
chunks = chunk_fixed_size(text, chunk_size=500, overlap=50)
doc_id = add_document("test.pdf", "pdf")
store_chunks(doc_id, chunks, strategy="fixed")
print(f"Stored {len(chunks)} chunks, document ID: {doc_id}")

Embedding 15 chunks using strategy 'fixed'...
Stored 15 chunks successfully.
Stored 15 chunks, document ID: 2


In [4]:
from app.retrieval.hybrid import hybrid_search, bm25_search
from app.db.vector_store import search_chunks

query = "Wikipedia editing rules"

print("=== DENSE ONLY ===")
dense = search_chunks(query, top_k=3)
for r in dense:
    print(f"  {r['similarity']:.3f} | {r['content'][:80]}...")

print("\n=== BM25 ONLY ===")
bm25 = bm25_search(query, top_k=3)
for r in bm25:
    print(f"  {r['bm25_score']:.3f} | {r['content'][:80]}...")

print("\n=== HYBRID (RRF) ===")
hybrid = hybrid_search(query, top_k=3)
for r in hybrid:
    print(f"  RRF:{r['rrf_score']:.4f} | {r['content'][:80]}...")

=== DENSE ONLY ===
  0.634 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiari...
  0.629 |  else is a faithful reproduction of 
the way the entry stood at the moment we or...
  0.613 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Jos...

=== BM25 ONLY ===
  2.289 |  so-called “Doctor” Dre never 
actually received a PhD.
D.A.R.E. America has its...
  1.200 | . Starting the blog was a no-brainer; our only concern 
was whether, after a few...
  1.132 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiari...

=== HYBRID (RRF) ===
Running hybrid search for: 'Wikipedia editing rules'
Dense results: 10
BM25 results: 10
Fused results: 3
  RRF:0.0323 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiari...
  RRF:0.0318 |  so-called “Doctor” Dre never 
actually received a PhD.
D.A.R.E. America has its...
  RRF:0.0315 |  else is a faithful reproduction of 
the way the entry sto

In [5]:
query = "Conor Lastowka"  # specific author name from the document

print("=== DENSE ONLY ===")
dense = search_chunks(query, top_k=3)
for r in dense:
    print(f"  {r['similarity']:.3f} | {r['content'][:100]}...")

print("\n=== BM25 ONLY ===")
bm25 = bm25_search(query, top_k=3)
for r in bm25:
    print(f"  {r['bm25_score']:.3f} | {r['content'][:100]}...")

print("\n=== HYBRID (RRF) ===")
hybrid = hybrid_search(query, top_k=3)
for r in hybrid:
    print(f"  RRF:{r['rrf_score']:.4f} | {r['content'][:100]}...")

=== DENSE ONLY ===
  0.618 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring ...
  0.564 |  else is a faithful reproduction of 
the way the entry stood at the moment we or our informants 
enc...
  0.550 | . Starting the blog was a no-brainer; our only concern 
was whether, after a few months of our daily...

=== BM25 ONLY ===
  5.804 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring ...
  1.677 |  else is a faithful reproduction of 
the way the entry stood at the moment we or our informants 
enc...
  0.000 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiarized a history report...

=== HYBRID (RRF) ===
Running hybrid search for: 'Conor Lastowka'
Dense results: 10
BM25 results: 10
Fused results: 3
  RRF:0.0328 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhlinger
Boring ...
  RRF:0.0323 |  else is a faithful reprod

In [2]:
import sys
sys.path.append("..")

from app.retrieval.hybrid import hybrid_search, hybrid_search_reranked
from app.retrieval.reranker import rerank
from app.db.vector_store import search_chunks
from app.retrieval.hybrid import bm25_search

query = "Wikipedia editing rules"

print("=== HYBRID (before reranking) ===")
candidates = hybrid_search(query, top_k=5)
for r in candidates:
    print(f"  RRF:{r['rrf_score']:.4f} | {r['content'][:80]}...")

print("\n=== AFTER RERANKING ===")
reranked = hybrid_search_reranked(query, top_k=3)
for r in reranked:
    print(f"  Rerank:{r['rerank_score']:.3f} | {r['content'][:80]}...")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

C:\Users\Hp\OneDrive\Desktop\rag-eval-system\backend\venv\lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hp\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

=== HYBRID (before reranking) ===
Running hybrid search for: 'Wikipedia editing rules'
Dense results: 10
BM25 results: 10
Fused results: 5
  RRF:0.0323 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiari...
  RRF:0.0318 |  so-called “Doctor” Dre never 
actually received a PhD.
D.A.R.E. America has its...
  RRF:0.0315 |  else is a faithful reproduction of 
the way the entry stood at the moment we or...
  RRF:0.0315 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Jos...
  RRF:0.0313 | . Starting the blog was a no-brainer; our only concern 
was whether, after a few...

=== AFTER RERANKING ===
Running hybrid search for: 'Wikipedia editing rules'
Dense results: 10
BM25 results: 10
Fused results: 5
Candidates before reranking: 5
Results after reranking: 3
  Rerank:-3.111 | 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiari...
  Rerank:-3.238 | 

[Citation Needed]
The Best of Wikipedia’s Worst Writing


In [3]:
import sys
sys.path.append("..")

from app.retrieval.hybrid import hybrid_search_reranked
from app.generation.generator import generate_answer

query = "What is Wikipedia and why do people edit it?"

# Step 1: retrieve and rerank
chunks = hybrid_search_reranked(query, top_k=3)
print(f"Retrieved {len(chunks)} chunks\n")

# Step 2: generate answer with citations
result = generate_answer(query, chunks)

print("=== ANSWER ===")
print(result["answer"])
print("\n=== SOURCES ===")
print(result["sources"])
print("\n=== CONTEXT USED ===")
for i, ctx in enumerate(result["context"], 1):
    print(f"\n[{i}] {ctx[:150]}...")

Running hybrid search for: 'What is Wikipedia and why do people edit it?'
Dense results: 10
BM25 results: 10
Fused results: 5
Candidates before reranking: 5
Results after reranking: 3
Retrieved 3 chunks

=== ANSWER ===
Wikipedia is an online encyclopedia that allows anyone to edit its content [1]. People edit it for various reasons, including to contribute knowledge, settle arguments, or simply to write about obscure topics, sometimes resulting in poorly written articles [1]. The fact that anyone can edit Wikipedia means that individuals with diverse interests and expertise, including those who are passionate about specific topics like pro wrestling, can contribute to its content [1].

=== SOURCES ===
['test.pdf']

=== CONTEXT USED ===

[1] 
Introduction
Wikipedia. Whether you’ve used it to settle an argument, 
plagiarized a history report from it, or simply replaced the 
entire text of t...

[2] 

[Citation Needed]
The Best of Wikipedia’s Worst Writing
Conor Lastowka and Josh Fruhling

In [1]:
import sys
sys.path.append("..")

from app.retrieval.hybrid import hybrid_search_reranked
from app.generation.generator import generate_answer
from app.evaluation.ragas_eval import evaluate_single

# use one question from your eval set that we know has content in the document
question = "Who copy-edited the book Citation Needed?"
ground_truth = "The book was copy-edited by Lauren Lastowka."

# Step 1: retrieve + rerank
chunks = hybrid_search_reranked(question, top_k=3)

# Step 2: generate answer
result = generate_answer(question, chunks)
print("Answer:", result["answer"])
print()

# Step 3: evaluate
print("Running RAGAS evaluation...")
scores = evaluate_single(
    question=question,
    answer=result["answer"],
    contexts=result["context"],
    ground_truth=ground_truth,
)

print("\n=== EVAL SCORES ===")
for metric, score in scores.items():
    print(f"  {metric}: {score}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [1]:
import sys
sys.path.append("..")

from app.retrieval.hybrid import hybrid_search_reranked
from app.generation.generator import generate_answer
from app.evaluation.evaluator import evaluate_all

question = "Who copy-edited the book Citation Needed?"
ground_truth = "The book was copy-edited by Lauren Lastowka."

# retrieve + generate
chunks = hybrid_search_reranked(question, top_k=3)
result = generate_answer(question, chunks)

print("Answer:", result["answer"])
print()

# evaluate
scores = evaluate_all(
    question=question,
    answer=result["answer"],
    contexts=result["context"],
    ground_truth=ground_truth,
)

print("\n=== SCORES ===")
print(f"Faithfulness: {scores['faithfulness']['score']}")
print(f"Answer Relevancy: {scores['answer_relevancy']['score']}")
print(f"Context Precision: {scores['context_precision']['score']}")
print(f"Context Recall: {scores['context_recall']['score']}")
print(f"Hallucination: {scores['hallucination']['has_hallucination']}")
print(f"Explanation: {scores['hallucination']['explanation']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Running hybrid search for: 'Who copy-edited the book Citation Needed?'
Dense results: 10
BM25 results: 10
Fused results: 5
Candidates before reranking: 5
Results after reranking: 3
Answer: Lauren Lastowka copy-edited the book [1].

Evaluating: 'Who copy-edited the book Citation Needed?...'
  Running faithfulness...
  Running answer relevancy...
  Running hallucination check...
  Running context precision...
  Running context recall...

=== SCORES ===
Faithfulness: 1.0
Answer Relevancy: 0.923
Context Precision: 0.333
Context Recall: 1.0
Hallucination: False
Explanation: All claims are supported by the context


In [2]:
import json
import sys
sys.path.append("..")

from app.retrieval.hybrid import hybrid_search_reranked
from app.generation.generator import generate_answer
from app.evaluation.evaluator import evaluate_all

# load eval set
with open("../../data/eval_set.json", "r", encoding="utf-8") as f:
    eval_set = json.load(f)

print(f"Running evaluation on {len(eval_set)} questions...\n")

all_results = []

for i, item in enumerate(eval_set):
    question = item["question"]
    ground_truth = item["ground_truth"]
    
    print(f"\n[{i+1}/{len(eval_set)}] {question[:60]}...")
    
    try:
        # retrieve + generate
        chunks = hybrid_search_reranked(question, top_k=3)
        result = generate_answer(question, chunks)
        
        # evaluate
        scores = evaluate_all(
            question=question,
            answer=result["answer"],
            contexts=result["context"],
            ground_truth=ground_truth,
        )
        
        all_results.append({
            "question": question,
            "answer": result["answer"],
            "ground_truth": ground_truth,
            "scores": {
                "faithfulness": scores["faithfulness"]["score"],
                "answer_relevancy": scores["answer_relevancy"]["score"],
                "context_precision": scores["context_precision"]["score"],
                "context_recall": scores["context_recall"]["score"],
                "hallucination": scores["hallucination"]["has_hallucination"],
            }
        })
        
    except Exception as e:
        print(f"  ERROR: {e}")
        all_results.append({
            "question": question,
            "error": str(e)
        })

# compute averages
valid = [r for r in all_results if "scores" in r]
if valid:
    avg_faithfulness = sum(r["scores"]["faithfulness"] for r in valid) / len(valid)
    avg_relevancy = sum(r["scores"]["answer_relevancy"] for r in valid) / len(valid)
    avg_precision = sum(r["scores"]["context_precision"] for r in valid) / len(valid)
    avg_recall = sum(r["scores"]["context_recall"] for r in valid) / len(valid)
    hallucination_rate = sum(1 for r in valid if r["scores"]["hallucination"]) / len(valid)

    print(f"\n{'='*50}")
    print(f"EVALUATION RESULTS ({len(valid)}/{len(eval_set)} successful)")
    print(f"{'='*50}")
    print(f"Faithfulness:       {avg_faithfulness:.3f}")
    print(f"Answer Relevancy:   {avg_relevancy:.3f}")
    print(f"Context Precision:  {avg_precision:.3f}")
    print(f"Context Recall:     {avg_recall:.3f}")
    print(f"Hallucination Rate: {hallucination_rate:.1%}")

# save results
with open("../../data/eval_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nResults saved to data/eval_results.json")

Running evaluation on 16 questions...


[1/16] Who copy-edited the book 'Citation Needed'?...
Running hybrid search for: 'Who copy-edited the book 'Citation Needed'?'
Dense results: 10
BM25 results: 10
Fused results: 5
Candidates before reranking: 5
Results after reranking: 3
Evaluating: 'Who copy-edited the book 'Citation Needed'?...'
  Running faithfulness...
  Running answer relevancy...
  Running hallucination check...
  Running context precision...
  Running context recall...

[2/16] Which film director felt that O.J. Simpson would not be beli...
Running hybrid search for: 'Which film director felt that O.J. Simpson would not be believable as the killer in The Terminator?'
Dense results: 10
BM25 results: 10
Fused results: 5
Candidates before reranking: 5
Results after reranking: 3
Evaluating: 'Which film director felt that O.J. Simpson would n...'
  Running faithfulness...
  Running answer relevancy...
  Running hallucination check...
  Running context precision...
  Running conte